# Astoria Manpower Optimization Analysis
We analyse order effort and forecasts staffing requirements in order to identify how to effectively optimise manpower in Astoria.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Set plot style
plt.rcParams['figure.figsize'] = [12, 6]
print('Libraries Loaded.')

Libraries Loaded.


### 1. Load and Clean Data

In [ ]:
df = pd.read_csv('../data1/astoria_orders.csv', parse_dates=['booking_date','collection_date','delivery_date'])
print(f'Data loaded: {len(df)} orders.')
df[['order_id', 'collection_date', 'delivery_date', 'effort_score']].head()

Data loaded: 4500 orders.


,order_id,collection_date,delivery_date,effort_score
0,AST-00001,2024-08-28,2024-09-07,4
1,AST-00002,2025-06-23,2025-06-25,1
2,AST-00003,2024-09-11,2024-09-14,2
3,AST-00004,2024-04-16,2024-04-20,1
4,AST-00005,2025-01-17,2025-01-20,2


### 2. Daily Effort Distribution
We calculate the workload by distributing an order's effort score across the days it sits in the facility.

In [ ]:
all_dates = pd.date_range(start=df['collection_date'].min(), end=df['delivery_date'].max())
workload = pd.Series(0.0, index=all_dates)

for _, row in df.iterrows():
    start, end = row['collection_date'], row['delivery_date']
    days = (end - start).days
    effort = row['effort_score']
    
    if days > 0:
        # Divide effort by processing days
        daily_val = effort / (days + 1)
        workload[start:end] += daily_val
    else:
        workload[start] += effort

workload_df = workload.to_frame(name='daily_effort')
workload_df.plot(title='Daily Processing Workload Trend')
plt.show()

### 3. Forecasting
Predicting the next 14 days using Holt-Winters Exponential Smoothing.

In [ ]:
model = ExponentialSmoothing(workload_df['daily_effort'], seasonal='add', seasonal_periods=7).fit()
forecast = model.forecast(14)

plt.plot(workload_df['daily_effort'].tail(60), label='Historical (Last 60 Days)')
plt.plot(forecast, label='14-Day Forecast', color='orange', marker='o')
plt.title('Workload Forecast')
plt.legend()
plt.show()

### 4. Manpower Calculation
Assuming 1 staff member can handle 8 effort units per day.

In [ ]:
STAFF_CAPACITY = 8.0
workload_df['staff_needed'] = np.ceil(workload_df['daily_effort'] / STAFF_CAPACITY)

print(f'Minimum staff needed (Avg): {workload_df["staff_needed"].mean():.2f}')
print(f'Max staff needed (Peak): {workload_df["staff_needed"].max()}')